# 02 · Train + predict + score — baseline nnU-Net (BraTS2023 GLI, 600-case subset)

One GPU run that does everything (plan C2). Everything nnU-Net writes goes under `/tmp`
(≈1 TB) so `/kaggle/working`'s 20 GB never fills.

1. **Setup** — install nnU-Net, rebuild the 600-case symlink raw folder.
2. **Preprocess FRESH** the 600 subset into `/tmp` (~3–4 h on 4 CPUs). We never reuse the
   old preprocess kernel: its plan was computed on all 1061 cases, not this subset.
3. **Train** `nnUNetTrainer_50epochs`, fold 0 — but if preprocessing + 50 epochs +
   prediction won't fit in ~11 h, we stop early (using `checkpoint_best.pth`) so the run
   finishes cleanly instead of being killed at Kaggle's 12 h limit.
4. **Predict** GLI test (190) and Africa (95), each with all sequences and again with T1c
   zeroed out; save per-case uncertainty (mean entropy in the predicted whole tumour).
5. **Score** Dice + HD95 for whole tumour / tumour core / enhancing tumour → one CSV.
6. **Trim** `/kaggle/working` to the CSV, training log, progress.png, final checkpoint and
   the predicted masks (.nii.gz).

In [ ]:
# Step 1 — setup: install, timers, GPU
import os, sys, time, subprocess, glob, shutil, json, re
START = time.time()
BUDGET_S         = 11.0 * 3600     # self-abort margin below Kaggle's 12 h hard kill
PREDICT_BUDGET_S =  3.0 * 3600     # reserve for 4 prediction sets + scoring
os.environ["nnUNet_n_proc_DA"] = "4"   # Kaggle has 4 CPUs; cap DA workers (default 12 would thrash)
subprocess.run([sys.executable,"-m","pip","install","-q","nnunetv2","medpy"], check=True)
from importlib.metadata import version
print("nnunetv2", version("nnunetv2"), "| medpy", version("medpy"))
print("GPU:", subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip())

In [ ]:
TRAIN = [
    "BraTS-GLI-00000-000",
    "BraTS-GLI-00002-000",
    "BraTS-GLI-00006-000",
    "BraTS-GLI-00008-000",
    "BraTS-GLI-00008-001",
    "BraTS-GLI-00012-000",
    "BraTS-GLI-00014-000",
    "BraTS-GLI-00017-000",
    "BraTS-GLI-00019-000",
    "BraTS-GLI-00020-000",
    "BraTS-GLI-00021-000",
    "BraTS-GLI-00021-001",
    "BraTS-GLI-00024-000",
    "BraTS-GLI-00024-001",
    "BraTS-GLI-00025-000",
    "BraTS-GLI-00026-000",
    "BraTS-GLI-00028-000",
    "BraTS-GLI-00031-001",
    "BraTS-GLI-00033-000",
    "BraTS-GLI-00036-000",
    "BraTS-GLI-00044-000",
    "BraTS-GLI-00051-000",
    "BraTS-GLI-00053-001",
    "BraTS-GLI-00056-001",
    "BraTS-GLI-00058-001",
    "BraTS-GLI-00062-000",
    "BraTS-GLI-00066-000",
    "BraTS-GLI-00068-000",
    "BraTS-GLI-00071-000",
    "BraTS-GLI-00081-000",
    "BraTS-GLI-00088-000",
    "BraTS-GLI-00088-001",
    "BraTS-GLI-00095-000",
    "BraTS-GLI-00095-001",
    "BraTS-GLI-00096-001",
    "BraTS-GLI-00098-000",
    "BraTS-GLI-00098-001",
    "BraTS-GLI-00099-000",
    "BraTS-GLI-00102-000",
    "BraTS-GLI-00107-000",
    "BraTS-GLI-00108-000",
    "BraTS-GLI-00109-000",
    "BraTS-GLI-00111-000",
    "BraTS-GLI-00113-000",
    "BraTS-GLI-00115-000",
    "BraTS-GLI-00116-000",
    "BraTS-GLI-00117-000",
    "BraTS-GLI-00120-000",
    "BraTS-GLI-00121-000",
    "BraTS-GLI-00123-000",
    "BraTS-GLI-00126-000",
    "BraTS-GLI-00127-000",
    "BraTS-GLI-00131-000",
    "BraTS-GLI-00132-000",
    "BraTS-GLI-00133-000",
    "BraTS-GLI-00136-000",
    "BraTS-GLI-00137-000",
    "BraTS-GLI-00144-000",
    "BraTS-GLI-00149-000",
    "BraTS-GLI-00152-000",
    "BraTS-GLI-00154-000",
    "BraTS-GLI-00157-000",
    "BraTS-GLI-00158-000",
    "BraTS-GLI-00159-000",
    "BraTS-GLI-00160-000",
    "BraTS-GLI-00166-000",
    "BraTS-GLI-00167-000",
    "BraTS-GLI-00170-000",
    "BraTS-GLI-00172-000",
    "BraTS-GLI-00176-000",
    "BraTS-GLI-00177-000",
    "BraTS-GLI-00185-000",
    "BraTS-GLI-00186-000",
    "BraTS-GLI-00191-000",
    "BraTS-GLI-00201-000",
    "BraTS-GLI-00203-000",
    "BraTS-GLI-00204-000",
    "BraTS-GLI-00207-000",
    "BraTS-GLI-00209-000",
    "BraTS-GLI-00214-000",
    "BraTS-GLI-00216-000",
    "BraTS-GLI-00217-000",
    "BraTS-GLI-00227-000",
    "BraTS-GLI-00228-000",
    "BraTS-GLI-00233-000",
    "BraTS-GLI-00237-000",
    "BraTS-GLI-00239-000",
    "BraTS-GLI-00240-000",
    "BraTS-GLI-00243-000",
    "BraTS-GLI-00249-000",
    "BraTS-GLI-00250-000",
    "BraTS-GLI-00253-000",
    "BraTS-GLI-00254-000",
    "BraTS-GLI-00258-000",
    "BraTS-GLI-00260-000",
    "BraTS-GLI-00271-000",
    "BraTS-GLI-00274-000",
    "BraTS-GLI-00280-000",
    "BraTS-GLI-00281-000",
    "BraTS-GLI-00282-000",
    "BraTS-GLI-00285-000",
    "BraTS-GLI-00288-000",
    "BraTS-GLI-00290-000",
    "BraTS-GLI-00292-000",
    "BraTS-GLI-00293-000",
    "BraTS-GLI-00298-000",
    "BraTS-GLI-00299-000",
    "BraTS-GLI-00301-000",
    "BraTS-GLI-00304-000",
    "BraTS-GLI-00313-000",
    "BraTS-GLI-00318-000",
    "BraTS-GLI-00321-000",
    "BraTS-GLI-00322-000",
    "BraTS-GLI-00325-000",
    "BraTS-GLI-00329-000",
    "BraTS-GLI-00331-000",
    "BraTS-GLI-00336-000",
    "BraTS-GLI-00338-000",
    "BraTS-GLI-00346-000",
    "BraTS-GLI-00349-000",
    "BraTS-GLI-00350-000",
    "BraTS-GLI-00366-000",
    "BraTS-GLI-00367-000",
    "BraTS-GLI-00371-000",
    "BraTS-GLI-00377-000",
    "BraTS-GLI-00378-000",
    "BraTS-GLI-00379-000",
    "BraTS-GLI-00380-000",
    "BraTS-GLI-00390-000",
    "BraTS-GLI-00395-000",
    "BraTS-GLI-00399-000",
    "BraTS-GLI-00400-000",
    "BraTS-GLI-00401-000",
    "BraTS-GLI-00402-000",
    "BraTS-GLI-00405-000",
    "BraTS-GLI-00406-000",
    "BraTS-GLI-00413-000",
    "BraTS-GLI-00416-000",
    "BraTS-GLI-00418-000",
    "BraTS-GLI-00425-000",
    "BraTS-GLI-00429-000",
    "BraTS-GLI-00441-000",
    "BraTS-GLI-00442-000",
    "BraTS-GLI-00443-000",
    "BraTS-GLI-00446-000",
    "BraTS-GLI-00449-000",
    "BraTS-GLI-00455-000",
    "BraTS-GLI-00457-000",
    "BraTS-GLI-00459-000",
    "BraTS-GLI-00464-000",
    "BraTS-GLI-00464-001",
    "BraTS-GLI-00469-001",
    "BraTS-GLI-00480-000",
    "BraTS-GLI-00483-000",
    "BraTS-GLI-00483-001",
    "BraTS-GLI-00485-000",
    "BraTS-GLI-00485-001",
    "BraTS-GLI-00491-000",
    "BraTS-GLI-00491-001",
    "BraTS-GLI-00495-000",
    "BraTS-GLI-00502-000",
    "BraTS-GLI-00504-000",
    "BraTS-GLI-00507-000",
    "BraTS-GLI-00512-000",
    "BraTS-GLI-00514-000",
    "BraTS-GLI-00514-001",
    "BraTS-GLI-00517-001",
    "BraTS-GLI-00524-000",
    "BraTS-GLI-00525-000",
    "BraTS-GLI-00528-000",
    "BraTS-GLI-00529-000",
    "BraTS-GLI-00530-000",
    "BraTS-GLI-00533-000",
    "BraTS-GLI-00537-000",
    "BraTS-GLI-00538-000",
    "BraTS-GLI-00539-000",
    "BraTS-GLI-00544-001",
    "BraTS-GLI-00550-001",
    "BraTS-GLI-00552-000",
    "BraTS-GLI-00554-000",
    "BraTS-GLI-00558-000",
    "BraTS-GLI-00559-000",
    "BraTS-GLI-00561-000",
    "BraTS-GLI-00567-000",
    "BraTS-GLI-00572-000",
    "BraTS-GLI-00574-000",
    "BraTS-GLI-00578-000",
    "BraTS-GLI-00580-000",
    "BraTS-GLI-00584-000",
    "BraTS-GLI-00587-000",
    "BraTS-GLI-00588-000",
    "BraTS-GLI-00589-000",
    "BraTS-GLI-00591-000",
    "BraTS-GLI-00596-000",
    "BraTS-GLI-00602-000",
    "BraTS-GLI-00602-001",
    "BraTS-GLI-00608-001",
    "BraTS-GLI-00616-000",
    "BraTS-GLI-00618-000",
    "BraTS-GLI-00619-000",
    "BraTS-GLI-00620-000",
    "BraTS-GLI-00620-001",
    "BraTS-GLI-00623-000",
    "BraTS-GLI-00625-000",
    "BraTS-GLI-00626-000",
    "BraTS-GLI-00636-000",
    "BraTS-GLI-00636-001",
    "BraTS-GLI-00640-000",
    "BraTS-GLI-00641-000",
    "BraTS-GLI-00641-001",
    "BraTS-GLI-00649-000",
    "BraTS-GLI-00650-000",
    "BraTS-GLI-00651-000",
    "BraTS-GLI-00654-000",
    "BraTS-GLI-00654-001",
    "BraTS-GLI-00655-000",
    "BraTS-GLI-00661-000",
    "BraTS-GLI-00668-000",
    "BraTS-GLI-00674-000",
    "BraTS-GLI-00675-000",
    "BraTS-GLI-00676-001",
    "BraTS-GLI-00677-000",
    "BraTS-GLI-00679-000",
    "BraTS-GLI-00684-000",
    "BraTS-GLI-00687-000",
    "BraTS-GLI-00688-000",
    "BraTS-GLI-00689-000",
    "BraTS-GLI-00689-001",
    "BraTS-GLI-00692-000",
    "BraTS-GLI-00692-001",
    "BraTS-GLI-00693-000",
    "BraTS-GLI-00697-000",
    "BraTS-GLI-00706-000",
    "BraTS-GLI-00714-001",
    "BraTS-GLI-00716-000",
    "BraTS-GLI-00727-001",
    "BraTS-GLI-00729-000",
    "BraTS-GLI-00729-001",
    "BraTS-GLI-00730-000",
    "BraTS-GLI-00730-001",
    "BraTS-GLI-00731-001",
    "BraTS-GLI-00732-000",
    "BraTS-GLI-00733-001",
    "BraTS-GLI-00734-001",
    "BraTS-GLI-00735-001",
    "BraTS-GLI-00736-001",
    "BraTS-GLI-00739-000",
    "BraTS-GLI-00744-000",
    "BraTS-GLI-00750-000",
    "BraTS-GLI-00753-000",
    "BraTS-GLI-00756-001",
    "BraTS-GLI-00760-000",
    "BraTS-GLI-00772-000",
    "BraTS-GLI-00772-001",
    "BraTS-GLI-00773-000",
    "BraTS-GLI-00777-000",
    "BraTS-GLI-00787-000",
    "BraTS-GLI-00792-000",
    "BraTS-GLI-00799-000",
    "BraTS-GLI-00800-000",
    "BraTS-GLI-00804-000",
    "BraTS-GLI-00807-000",
    "BraTS-GLI-00808-000",
    "BraTS-GLI-00814-000",
    "BraTS-GLI-00823-000",
    "BraTS-GLI-00824-000",
    "BraTS-GLI-00828-000",
    "BraTS-GLI-00830-000",
    "BraTS-GLI-00837-000",
    "BraTS-GLI-00838-000",
    "BraTS-GLI-00839-000",
    "BraTS-GLI-01001-000",
    "BraTS-GLI-01009-000",
    "BraTS-GLI-01011-000",
    "BraTS-GLI-01014-000",
    "BraTS-GLI-01015-000",
    "BraTS-GLI-01017-000",
    "BraTS-GLI-01020-000",
    "BraTS-GLI-01022-000",
    "BraTS-GLI-01024-000",
    "BraTS-GLI-01027-001",
    "BraTS-GLI-01033-000",
    "BraTS-GLI-01035-000",
    "BraTS-GLI-01036-000",
    "BraTS-GLI-01037-000",
    "BraTS-GLI-01039-000",
    "BraTS-GLI-01044-000",
    "BraTS-GLI-01054-000",
    "BraTS-GLI-01055-000",
    "BraTS-GLI-01057-000",
    "BraTS-GLI-01058-000",
    "BraTS-GLI-01061-000",
    "BraTS-GLI-01062-000",
    "BraTS-GLI-01063-000",
    "BraTS-GLI-01064-000",
    "BraTS-GLI-01066-000",
    "BraTS-GLI-01071-000",
    "BraTS-GLI-01074-000",
    "BraTS-GLI-01075-000",
    "BraTS-GLI-01079-000",
    "BraTS-GLI-01083-000",
    "BraTS-GLI-01088-000",
    "BraTS-GLI-01089-000",
    "BraTS-GLI-01092-000",
    "BraTS-GLI-01094-000",
    "BraTS-GLI-01096-000",
    "BraTS-GLI-01097-000",
    "BraTS-GLI-01099-000",
    "BraTS-GLI-01100-000",
    "BraTS-GLI-01103-000",
    "BraTS-GLI-01104-000",
    "BraTS-GLI-01106-000",
    "BraTS-GLI-01107-000",
    "BraTS-GLI-01113-000",
    "BraTS-GLI-01115-000",
    "BraTS-GLI-01116-000",
    "BraTS-GLI-01118-000",
    "BraTS-GLI-01121-000",
    "BraTS-GLI-01122-000",
    "BraTS-GLI-01123-000",
    "BraTS-GLI-01125-000",
    "BraTS-GLI-01126-000",
    "BraTS-GLI-01128-000",
    "BraTS-GLI-01129-000",
    "BraTS-GLI-01131-000",
    "BraTS-GLI-01132-000",
    "BraTS-GLI-01137-000",
    "BraTS-GLI-01141-000",
    "BraTS-GLI-01145-000",
    "BraTS-GLI-01151-000",
    "BraTS-GLI-01153-000",
    "BraTS-GLI-01157-000",
    "BraTS-GLI-01160-000",
    "BraTS-GLI-01163-000",
    "BraTS-GLI-01164-000",
    "BraTS-GLI-01168-000",
    "BraTS-GLI-01171-000",
    "BraTS-GLI-01173-000",
    "BraTS-GLI-01178-000",
    "BraTS-GLI-01179-000",
    "BraTS-GLI-01181-000",
    "BraTS-GLI-01182-000",
    "BraTS-GLI-01185-000",
    "BraTS-GLI-01186-000",
    "BraTS-GLI-01194-000",
    "BraTS-GLI-01195-000",
    "BraTS-GLI-01197-000",
    "BraTS-GLI-01198-000",
    "BraTS-GLI-01203-000",
    "BraTS-GLI-01205-000",
    "BraTS-GLI-01207-000",
    "BraTS-GLI-01209-000",
    "BraTS-GLI-01210-000",
    "BraTS-GLI-01213-000",
    "BraTS-GLI-01221-000",
    "BraTS-GLI-01223-000",
    "BraTS-GLI-01224-000",
    "BraTS-GLI-01229-000",
    "BraTS-GLI-01231-000",
    "BraTS-GLI-01235-000",
    "BraTS-GLI-01236-000",
    "BraTS-GLI-01238-000",
    "BraTS-GLI-01239-000",
    "BraTS-GLI-01242-000",
    "BraTS-GLI-01243-000",
    "BraTS-GLI-01247-000",
    "BraTS-GLI-01254-000",
    "BraTS-GLI-01255-000",
    "BraTS-GLI-01256-000",
    "BraTS-GLI-01257-000",
    "BraTS-GLI-01258-000",
    "BraTS-GLI-01260-000",
    "BraTS-GLI-01261-000",
    "BraTS-GLI-01263-000",
    "BraTS-GLI-01264-000",
    "BraTS-GLI-01270-000",
    "BraTS-GLI-01272-000",
    "BraTS-GLI-01273-000",
    "BraTS-GLI-01275-000",
    "BraTS-GLI-01276-000",
    "BraTS-GLI-01278-000",
    "BraTS-GLI-01283-000",
    "BraTS-GLI-01284-000",
    "BraTS-GLI-01285-000",
    "BraTS-GLI-01290-000",
    "BraTS-GLI-01291-000",
    "BraTS-GLI-01292-000",
    "BraTS-GLI-01294-000",
    "BraTS-GLI-01295-000",
    "BraTS-GLI-01297-000",
    "BraTS-GLI-01299-000",
    "BraTS-GLI-01300-000",
    "BraTS-GLI-01307-000",
    "BraTS-GLI-01310-000",
    "BraTS-GLI-01316-000",
    "BraTS-GLI-01317-000",
    "BraTS-GLI-01323-000",
    "BraTS-GLI-01324-000",
    "BraTS-GLI-01326-000",
    "BraTS-GLI-01330-000",
    "BraTS-GLI-01332-000",
    "BraTS-GLI-01337-000",
    "BraTS-GLI-01339-000",
    "BraTS-GLI-01345-000",
    "BraTS-GLI-01347-000",
    "BraTS-GLI-01348-000",
    "BraTS-GLI-01349-000",
    "BraTS-GLI-01350-000",
    "BraTS-GLI-01354-000",
    "BraTS-GLI-01357-000",
    "BraTS-GLI-01364-000",
    "BraTS-GLI-01365-000",
    "BraTS-GLI-01367-000",
    "BraTS-GLI-01370-000",
    "BraTS-GLI-01371-000",
    "BraTS-GLI-01372-000",
    "BraTS-GLI-01374-000",
    "BraTS-GLI-01375-000",
    "BraTS-GLI-01376-000",
    "BraTS-GLI-01381-000",
    "BraTS-GLI-01383-000",
    "BraTS-GLI-01387-000",
    "BraTS-GLI-01388-000",
    "BraTS-GLI-01391-000",
    "BraTS-GLI-01393-000",
    "BraTS-GLI-01395-000",
    "BraTS-GLI-01397-000",
    "BraTS-GLI-01398-000",
    "BraTS-GLI-01399-000",
    "BraTS-GLI-01400-000",
    "BraTS-GLI-01402-000",
    "BraTS-GLI-01403-000",
    "BraTS-GLI-01405-000",
    "BraTS-GLI-01406-000",
    "BraTS-GLI-01411-000",
    "BraTS-GLI-01415-000",
    "BraTS-GLI-01416-000",
    "BraTS-GLI-01419-000",
    "BraTS-GLI-01422-000",
    "BraTS-GLI-01423-000",
    "BraTS-GLI-01427-000",
    "BraTS-GLI-01431-000",
    "BraTS-GLI-01432-000",
    "BraTS-GLI-01433-000",
    "BraTS-GLI-01436-000",
    "BraTS-GLI-01437-000",
    "BraTS-GLI-01439-000",
    "BraTS-GLI-01440-000",
    "BraTS-GLI-01443-000",
    "BraTS-GLI-01444-000",
    "BraTS-GLI-01450-000",
    "BraTS-GLI-01453-000",
    "BraTS-GLI-01459-000",
    "BraTS-GLI-01460-000",
    "BraTS-GLI-01462-000",
    "BraTS-GLI-01463-000",
    "BraTS-GLI-01466-000",
    "BraTS-GLI-01468-000",
    "BraTS-GLI-01470-000",
    "BraTS-GLI-01474-000",
    "BraTS-GLI-01475-000",
    "BraTS-GLI-01481-000",
    "BraTS-GLI-01482-000",
    "BraTS-GLI-01485-000",
    "BraTS-GLI-01486-000",
    "BraTS-GLI-01488-000",
    "BraTS-GLI-01490-000",
    "BraTS-GLI-01494-000",
    "BraTS-GLI-01496-000",
    "BraTS-GLI-01498-000",
    "BraTS-GLI-01499-000",
    "BraTS-GLI-01501-000",
    "BraTS-GLI-01502-000",
    "BraTS-GLI-01504-000",
    "BraTS-GLI-01505-000",
    "BraTS-GLI-01507-000",
    "BraTS-GLI-01508-000",
    "BraTS-GLI-01509-000",
    "BraTS-GLI-01510-000",
    "BraTS-GLI-01511-000",
    "BraTS-GLI-01512-000",
    "BraTS-GLI-01515-000",
    "BraTS-GLI-01516-000",
    "BraTS-GLI-01517-000",
    "BraTS-GLI-01519-000",
    "BraTS-GLI-01520-000",
    "BraTS-GLI-01523-000",
    "BraTS-GLI-01524-000",
    "BraTS-GLI-01526-000",
    "BraTS-GLI-01529-000",
    "BraTS-GLI-01530-000",
    "BraTS-GLI-01533-000",
    "BraTS-GLI-01536-000",
    "BraTS-GLI-01537-000",
    "BraTS-GLI-01610-000",
    "BraTS-GLI-01657-000",
    "BraTS-GLI-01660-000",
    "BraTS-GLI-01662-000",
    "BraTS-GLI-01664-000",
    "BraTS-GLI-01665-000",
]

VAL = [
    "BraTS-GLI-00005-000",
    "BraTS-GLI-00035-000",
    "BraTS-GLI-00084-000",
    "BraTS-GLI-00090-000",
    "BraTS-GLI-00097-000",
    "BraTS-GLI-00101-000",
    "BraTS-GLI-00140-000",
    "BraTS-GLI-00147-000",
    "BraTS-GLI-00199-000",
    "BraTS-GLI-00251-000",
    "BraTS-GLI-00259-000",
    "BraTS-GLI-00273-000",
    "BraTS-GLI-00283-000",
    "BraTS-GLI-00284-000",
    "BraTS-GLI-00296-000",
    "BraTS-GLI-00297-000",
    "BraTS-GLI-00300-000",
    "BraTS-GLI-00370-000",
    "BraTS-GLI-00388-000",
    "BraTS-GLI-00389-000",
    "BraTS-GLI-00391-000",
    "BraTS-GLI-00419-000",
    "BraTS-GLI-00421-000",
    "BraTS-GLI-00440-000",
    "BraTS-GLI-00451-000",
    "BraTS-GLI-00468-000",
    "BraTS-GLI-00479-000",
    "BraTS-GLI-00496-000",
    "BraTS-GLI-00499-001",
    "BraTS-GLI-00510-001",
    "BraTS-GLI-00511-001",
    "BraTS-GLI-00516-000",
    "BraTS-GLI-00518-000",
    "BraTS-GLI-00542-000",
    "BraTS-GLI-00545-000",
    "BraTS-GLI-00555-000",
    "BraTS-GLI-00555-001",
    "BraTS-GLI-00557-000",
    "BraTS-GLI-00563-000",
    "BraTS-GLI-00576-000",
    "BraTS-GLI-00583-000",
    "BraTS-GLI-00594-000",
    "BraTS-GLI-00605-000",
    "BraTS-GLI-00606-000",
    "BraTS-GLI-00611-000",
    "BraTS-GLI-00611-001",
    "BraTS-GLI-00613-001",
    "BraTS-GLI-00615-000",
    "BraTS-GLI-00652-000",
    "BraTS-GLI-00680-001",
    "BraTS-GLI-00683-000",
    "BraTS-GLI-00683-001",
    "BraTS-GLI-00686-000",
    "BraTS-GLI-00707-000",
    "BraTS-GLI-00715-000",
    "BraTS-GLI-00725-001",
    "BraTS-GLI-00740-000",
    "BraTS-GLI-00751-000",
    "BraTS-GLI-00751-001",
    "BraTS-GLI-00764-000",
    "BraTS-GLI-00784-000",
    "BraTS-GLI-01003-000",
    "BraTS-GLI-01034-000",
    "BraTS-GLI-01110-000",
    "BraTS-GLI-01111-000",
    "BraTS-GLI-01112-000",
    "BraTS-GLI-01142-000",
    "BraTS-GLI-01143-000",
    "BraTS-GLI-01161-000",
    "BraTS-GLI-01174-000",
    "BraTS-GLI-01183-000",
    "BraTS-GLI-01184-000",
    "BraTS-GLI-01187-000",
    "BraTS-GLI-01190-000",
    "BraTS-GLI-01193-000",
    "BraTS-GLI-01199-000",
    "BraTS-GLI-01201-000",
    "BraTS-GLI-01202-000",
    "BraTS-GLI-01204-000",
    "BraTS-GLI-01206-000",
    "BraTS-GLI-01217-000",
    "BraTS-GLI-01225-000",
    "BraTS-GLI-01241-000",
    "BraTS-GLI-01246-000",
    "BraTS-GLI-01248-000",
    "BraTS-GLI-01280-000",
    "BraTS-GLI-01282-000",
    "BraTS-GLI-01288-000",
    "BraTS-GLI-01314-000",
    "BraTS-GLI-01336-000",
    "BraTS-GLI-01344-000",
    "BraTS-GLI-01401-000",
    "BraTS-GLI-01412-000",
    "BraTS-GLI-01424-000",
    "BraTS-GLI-01434-000",
    "BraTS-GLI-01458-000",
    "BraTS-GLI-01465-000",
    "BraTS-GLI-01467-000",
    "BraTS-GLI-01480-000",
    "BraTS-GLI-01658-000",
]

SUBSET = TRAIN + VAL
print("subset:", len(TRAIN), "train +", len(VAL), "val =", len(SUBSET))

In [ ]:
# Step 2 — locate dataset mount, env vars (all under /tmp except results), rebuild 600-case raw
MOUNT = None
for cand in glob.glob("/kaggle/input/**/dataset.json", recursive=True):
    d = os.path.dirname(cand)
    if os.path.isdir(os.path.join(d, "imagesTr")):
        MOUNT = d; break
assert MOUNT, "dataset mount not found"
print("MOUNT =", MOUNT)

os.environ["nnUNet_raw"]          = "/tmp/nnUNet_raw"
os.environ["nnUNet_preprocessed"] = "/tmp/nnUNet_preprocessed"
os.environ["nnUNet_results"]      = "/kaggle/working/nnUNet_results"
for k in ["nnUNet_raw","nnUNet_preprocessed","nnUNet_results"]:
    os.makedirs(os.environ[k], exist_ok=True)

RAW = "/tmp/nnUNet_raw/Dataset501_BraTS2023GLI"
for f in ["imagesTr","labelsTr"]:
    os.makedirs(os.path.join(RAW, f), exist_ok=True)

def link_one(src, link):
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    if os.path.isfile(src):
        os.symlink(src, link); return "file"
    if os.path.isdir(src):                       # nested X.nii/inner.nii
        inner = os.listdir(src)
        assert len(inner) == 1, f"{src}: {inner[:3]}"
        os.symlink(os.path.join(src, inner[0]), link); return "dir"
    raise FileNotFoundError(src)

n_img = n_lbl = 0
for cid in SUBSET:
    for ch in ("0000","0001","0002","0003"):
        link_one(os.path.join(MOUNT,"imagesTr",f"{cid}_{ch}.nii"),
                 os.path.join(RAW,"imagesTr",f"{cid}_{ch}.nii")); n_img += 1
    link_one(os.path.join(MOUNT,"labelsTr",f"{cid}.nii"),
             os.path.join(RAW,"labelsTr",f"{cid}.nii")); n_lbl += 1
assert n_img == 2400 and n_lbl == 600, (n_img, n_lbl)

ds = json.load(open(os.path.join(MOUNT,"dataset.json")))
ds["file_ending"] = ".nii"; ds["numTraining"] = 600
json.dump(ds, open(os.path.join(RAW,"dataset.json"),"w"), indent=2)
print("raw rebuilt: images", n_img, "labels", n_lbl, "| channels", ds["channel_names"])

In [ ]:
# Step 3 — preprocess the 600 subset FRESH into /tmp (plan computed on these 600 only)
PP = "/tmp/nnUNet_preprocessed/Dataset501_BraTS2023GLI"
def n_b2nd(d):
    g = os.path.join(d, "nnUNetPlans_3d_fullres")
    return len([x for x in os.listdir(g) if x.endswith(".b2nd") and not x.endswith("_seg.b2nd")]) if os.path.isdir(g) else 0

cmd = ["nnUNetv2_plan_and_preprocess","-d","501","-c","3d_fullres","--verify_dataset_integrity","-np","4"]
print("preprocessing 600 cases (expect ~3-4 h on 4 CPUs):", " ".join(cmd), flush=True)
t0 = time.time()
p = subprocess.run(cmd, env=os.environ, capture_output=True, text=True)
print("returncode", p.returncode, "in", round((time.time()-t0)/3600,2), "h")
print("\n".join(p.stdout.splitlines()[-25:]))
print("\n".join(p.stderr.splitlines()[-15:]))
assert p.returncode == 0, "plan_and_preprocess failed"

json.dump([{"train": TRAIN, "val": VAL}], open(os.path.join(PP,"splits_final.json"),"w"), indent=2)
assert n_b2nd(PP) >= 600, f"only {n_b2nd(PP)} cases preprocessed"
print("preprocessed cases:", n_b2nd(PP))
print("=== df -h /tmp /kaggle/working (after preprocessing) ===")
print(subprocess.run(["df","-h","/tmp","/kaggle/working"], capture_output=True, text=True).stdout)
print("elapsed after preprocessing:", round((time.time()-START)/3600,2), "h")

In [ ]:
# Step 4 — TRAIN fold 0 (built-in nnUNetTrainer_50epochs), cutting epochs if the ~11 h budget would be blown
MODEL = os.path.join(os.environ["nnUNet_results"], "Dataset501_BraTS2023GLI",
                     "nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres")
FOLD0 = os.path.join(MODEL, "fold_0")

cmd = ["nnUNetv2_train","501","3d_fullres","0","-tr","nnUNetTrainer_50epochs"]
print("running:", " ".join(cmd), flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, env=os.environ)
time_re  = re.compile(r"Epoch time:\s*([\d.]+)\s*s")
start_re = re.compile(r"\bEpoch (\d+)\b")

epoch_times, decided, cut, TARGET = [], False, False, 50
for line in proc.stdout:
    print(line, end="")
    mt = time_re.search(line)
    if mt:
        epoch_times.append(float(mt.group(1)))
        ne = len(epoch_times)
        if ne <= 2:
            print(f"[TIMER] epoch {ne-1} took {epoch_times[-1]:.1f} s", flush=True)
        if ne >= 2 and not decided:
            decided = True
            per   = sum(epoch_times) / ne
            now   = time.time() - START
            train_budget = BUDGET_S - now - PREDICT_BUDGET_S + sum(epoch_times)
            n_fit = int(train_budget // per)
            if n_fit >= 50:
                TARGET, cut = 50, False
                print(f"[BUDGET] ~{per:.0f}s/epoch -> full 50 epochs fit "
                      f"(train ~{per*50/3600:.2f} h). Running all 50.", flush=True)
            else:
                TARGET, cut = max(5, n_fit), True
                print(f"[BUDGET] ~{per:.0f}s/epoch -> only ~{n_fit} epochs fit in the budget; "
                      f"CUTTING to {TARGET} epochs to avoid the 12 h kill.", flush=True)
        continue
    ms = start_re.search(line)
    if ms and cut and int(ms.group(1)) >= TARGET:
        # next epoch starting => epoch TARGET-1's on_epoch_end (incl. checkpoint_best save) is done
        print(f"[BUDGET] {TARGET} epochs done & checkpointed; stopping training.", flush=True)
        proc.terminate(); break

if cut:
    try: proc.wait(timeout=180)
    except Exception: proc.kill()
else:
    proc.wait(); assert proc.returncode == 0, "training failed"

CHK = "checkpoint_final.pth"
if cut or not os.path.exists(os.path.join(FOLD0, "checkpoint_final.pth")):
    for cand in ("checkpoint_best.pth","checkpoint_latest.pth"):
        if os.path.exists(os.path.join(FOLD0, cand)): CHK = cand; break
assert os.path.exists(os.path.join(FOLD0, CHK)), f"no usable checkpoint in {FOLD0}: {os.listdir(FOLD0)}"
print("fold_0:", sorted(os.listdir(FOLD0)))
print("epochs run:", len(epoch_times), "| TARGET:", TARGET, "| cut:", cut, "| checkpoint:", CHK)
print("elapsed after training:", round((time.time()-START)/3600,2), "h")

In [ ]:
# Step 5 — build the four prediction input folders (.nii), zeroing T1c (_0001) for no_t1c
import nibabel as nib, numpy as np

GLI_IMG, GLI_LBL = os.path.join(MOUNT,"imagesTs_gli_test"), os.path.join(MOUNT,"labelsTs_gli_test")
AFR_IMG, AFR_LBL = os.path.join(MOUNT,"imagesTs_africa"),   os.path.join(MOUNT,"labelsTs_africa")

def resolve(src):                                  # nested dir -> its single inner file; else the file
    if os.path.isdir(src):
        inner = os.listdir(src); assert len(inner) == 1, src
        return os.path.join(src, inner[0])
    return src

def case_ids(img_dir):
    ids = set()
    for e in os.listdir(img_dir):
        name = e[:-4] if e.endswith(".nii") else e
        if name.endswith(("_0000","_0001","_0002","_0003")):
            ids.add(name[:-5])
    return sorted(ids)

def build_inputs(img_dir, cids, out_dir, drop_t1c):
    os.makedirs(out_dir, exist_ok=True)
    for cid in cids:
        for ch in ("0000","0001","0002","0003"):
            src = resolve(os.path.join(img_dir, f"{cid}_{ch}.nii"))
            dst = os.path.join(out_dir, f"{cid}_{ch}.nii")
            if os.path.islink(dst) or os.path.exists(dst): os.remove(dst)
            if drop_t1c and ch == "0001":
                im = nib.load(src)
                nib.save(nib.Nifti1Image(np.zeros(im.shape, dtype=im.get_data_dtype()),
                                         im.affine, im.header), dst)
            else:
                os.symlink(src, dst)
    return len(cids)

GLI = case_ids(GLI_IMG); AFR = case_ids(AFR_IMG)
print("GLI test cases:", len(GLI), "| Africa cases:", len(AFR))
# (set, condition, img_dir, cids, lbl_dir, in_dir, out_dir, drop_t1c)
SETS = [
    ("gli","full",    GLI_IMG, GLI, GLI_LBL, "/tmp/in/gli_full",     "/tmp/out/gli_full",     False),
    ("africa","full", AFR_IMG, AFR, AFR_LBL, "/tmp/in/africa_full",  "/tmp/out/africa_full",  False),
    ("gli","no_t1c",  GLI_IMG, GLI, GLI_LBL, "/tmp/in/gli_no_t1c",   "/tmp/out/gli_no_t1c",   True),
    ("africa","no_t1c",AFR_IMG,AFR, AFR_LBL, "/tmp/in/africa_no_t1c","/tmp/out/africa_no_t1c",True),
]
for s in SETS:
    print(f"built {s[0]}/{s[1]}: {build_inputs(s[2], s[3], s[5], s[7])} cases -> {s[5]}")

In [ ]:
# Step 6 — predict each set (final checkpoint, no TTA, save probs); per-case WT entropy; drop probs
EPS = 1e-7
def wt_entropy(prob):                              # prob: (regions, X,Y,Z); region 0 = whole tumour
    p = np.clip(prob, EPS, 1-EPS)
    H = -(p*np.log(p) + (1-p)*np.log(1-p)).mean(axis=0)   # per-voxel mean over region channels
    wt = prob[0] > 0.5                              # predicted whole-tumour region
    return float(H[wt].mean()) if wt.sum() else float("nan")

UNC = {}
for setname, cond, img_dir, cids, lbl_dir, in_dir, out_dir, drop in SETS:
    if time.time()-START > BUDGET_S + 0.4*3600:     # safety: don't risk the 12 h kill mid-predict
        print(f"!! out of time before {setname}/{cond}; skipping remaining sets"); break
    os.makedirs(out_dir, exist_ok=True)
    cmd = ["nnUNetv2_predict","-i",in_dir,"-o",out_dir,"-d","501","-c","3d_fullres",
           "-f","0","-tr","nnUNetTrainer_50epochs","-chk",CHK,"--disable_tta","--save_probabilities",
           "-npp","2","-nps","2"]   # cap export workers for Kaggle's 4 CPUs
    print(f"predict {setname}/{cond} ({len(cids)} cases) ...", flush=True)
    t0 = time.time()
    r = subprocess.run(cmd, env=os.environ, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:]); print(r.stderr[-2000:]); raise RuntimeError(f"predict failed {setname}/{cond}")
    print(f"  done in {(time.time()-t0)/60:.1f} min; entropy + dropping probs")
    d = {}
    for cid in cids:
        npz = os.path.join(out_dir, f"{cid}.npz")
        if os.path.exists(npz):
            d[cid] = wt_entropy(np.load(npz)["probabilities"]); os.remove(npz)
        pkl = os.path.join(out_dir, f"{cid}.pkl")
        if os.path.exists(pkl): os.remove(pkl)
    UNC[(setname,cond)] = d
    print(f"  entropy for {len(d)} cases; elapsed {round((time.time()-START)/3600,2)} h")

In [ ]:
# Step 7 — score: Dice + HD95 for WT (1+2+3), TC (1+3), ET (3) -> one CSV
import csv
from medpy.metric.binary import hd95 as _hd95
REGIONS = {"wt": (1,2,3), "tc": (1,3), "et": (3,)}
def mask(lbl, labs):
    m = np.zeros(lbl.shape, bool)
    for l in labs: m |= (lbl == l)
    return m
def dice(a, b):
    sa, sb = a.sum(), b.sum()
    if sa == 0 and sb == 0: return 1.0
    if sa == 0 or sb == 0:  return 0.0
    return 2.0*np.logical_and(a,b).sum()/(sa+sb)
def hd95(a, b):
    sa, sb = a.sum(), b.sum()
    if sa == 0 and sb == 0: return 0.0             # both empty -> perfect
    if sa == 0 or sb == 0:  return float("nan")    # exactly one empty -> undefined
    return float(_hd95(a, b, voxelspacing=(1,1,1)))

rows = []
for setname, cond, img_dir, cids, lbl_dir, in_dir, out_dir, drop in SETS:
    for cid in cids:
        pred_f = os.path.join(out_dir, f"{cid}.nii")
        if not os.path.exists(pred_f): continue
        pred = np.asarray(nib.load(pred_f).get_fdata()).astype(int)
        gt   = np.asarray(nib.load(resolve(os.path.join(lbl_dir, f"{cid}.nii"))).get_fdata()).astype(int)
        row = {"case_id":cid, "set":setname, "condition":cond,
               "uncertainty": UNC.get((setname,cond),{}).get(cid, float("nan"))}
        for rn, labs in REGIONS.items():
            p, g = mask(pred, labs), mask(gt, labs)
            row[f"dice_{rn}"] = dice(p, g)
            row[f"hd95_{rn}"] = hd95(p, g)
        rows.append(row)
    print("scored", setname, cond)

cols = ["case_id","set","condition","dice_wt","dice_tc","dice_et",
        "hd95_wt","hd95_tc","hd95_et","uncertainty"]
with open("/kaggle/working/results.csv","w",newline="") as f:
    w = csv.DictWriter(f, fieldnames=cols); w.writeheader()
    for r in rows: w.writerow({k:r.get(k,"") for k in cols})
print("wrote /kaggle/working/results.csv with", len(rows), "rows")
import statistics as st
for setname in ("gli","africa"):
    for cond in ("full","no_t1c"):
        sel = [r for r in rows if r["set"]==setname and r["condition"]==cond]
        if sel:
            print(f"{setname}/{cond}: n={len(sel)} meanDice WT={st.mean(r['dice_wt'] for r in sel):.3f} "
                  f"TC={st.mean(r['dice_tc'] for r in sel):.3f} ET={st.mean(r['dice_et'] for r in sel):.3f}")

In [ ]:
# Step 8 — trim /kaggle/working to only the wanted artifacts
import gzip
dst_pred = "/kaggle/working/predictions"; os.makedirs(dst_pred, exist_ok=True)
for setname, cond, img_dir, cids, lbl_dir, in_dir, out_dir, drop in SETS:
    sub = os.path.join(dst_pred, f"{setname}_{cond}"); os.makedirs(sub, exist_ok=True)
    for cid in cids:
        src = os.path.join(out_dir, f"{cid}.nii")
        if os.path.exists(src):
            with open(src,"rb") as fi, gzip.open(os.path.join(sub,f"{cid}.nii.gz"),"wb") as fo:
                shutil.copyfileobj(fi, fo)

for pat, outname in [("training_log_*.txt","training_log.txt"), ("progress.png","progress.png")]:
    hits = glob.glob(os.path.join(FOLD0, pat))
    if hits: shutil.copy(sorted(hits)[-1], os.path.join("/kaggle/working", outname))
shutil.copy(os.path.join(FOLD0, CHK), "/kaggle/working/checkpoint_final.pth")
for j in ("dataset.json","plans.json"):
    if os.path.exists(os.path.join(MODEL,j)): shutil.copy(os.path.join(MODEL,j), f"/kaggle/working/{j}")
shutil.rmtree(os.environ["nnUNet_results"], ignore_errors=True)

print("=== final /kaggle/working ===")
print(subprocess.run(["bash","-lc","ls -la /kaggle/working; echo; du -sh /kaggle/working; du -sh /kaggle/working/*"],
                     capture_output=True, text=True).stdout)
print("TOTAL notebook time:", round((time.time()-START)/3600,2), "h")